# Part 3 – Churn Prediction Model & Model Card

**Snapshot date:** 2025-09-30  
**Target:** `churn_next_60d` (no purchase Oct 1 – Nov 29 2025)  
**Feature source:** `rfm_modeling_snapshot.csv` 
**Models:** Logistic Regression (baseline) vs Gradient Boosting Classifier (final)


In [ ]:
from pathlib import Path
import json, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import GradientBoostingClassifier
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import OneHotEncoder, StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                      f1_score, roc_auc_score, average_precision_score,
                                      confusion_matrix, roc_curve, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid', palette='muted')

DATA = Path('data')
if not DATA.exists() or not any(DATA.glob('*.csv')):
    DATA = Path('dataset')

CHARTS = Path('outputs/charts')
CHARTS.mkdir(parents=True, exist_ok=True)
print('Data path:', DATA.resolve())

## 1. Load & Split Data

In [ ]:
snap = pd.read_csv(DATA / 'rfm_modeling_snapshot.csv')
snap['loyalty_tier']     = snap['loyalty_tier'].fillna('None')
snap['avg_rating_180d']  = snap['avg_rating_180d'].fillna(snap['avg_rating_180d'].median())

print(f'Total rows: {len(snap)}')
print('Split counts:')
print(snap['split'].value_counts().to_string())
print(f'\nOverall churn rate: {snap["churn_next_60d"].mean():.2%}')

## 2. Leakage Check

In [ ]:
# Columns that must NOT be used as features
TARGET   = 'churn_next_60d'
EXCLUDE  = ['customer_id', 'snapshot_date', TARGET, 'split']

# All remaining columns are pre-snapshot — verified via DATA_DICTIONARY.md
# No column names contain 'next', 'post', or 'future'
leaky_keywords = ['next', 'post_snapshot', 'future', 'after_snapshot']
leaky = [c for c in snap.columns if any(k in c.lower() for k in leaky_keywords) and c not in EXCLUDE]
print('Potential leakage columns (should be empty):', leaky)


In [ ]:
NUMERIC_FEATURES = [
    'recency_days', 'frequency_180d', 'monetary_180d',
    'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d',
    'category_diversity_180d', 'ticket_count_90d', 'negative_ticket_rate_90d',
    'avg_resolution_hours_90d', 'days_since_signup',
    'sessions_30d', 'product_views_30d', 'cart_adds_30d',
    'wishlist_adds_30d', 'abandoned_carts_30d',
    'email_opens_30d', 'campaign_clicks_30d', 'last_visit_days_ago'
]
CATEGORICAL_FEATURES = [
    'city_tier', 'age_group', 'acquisition_channel',
    'loyalty_tier', 'preferred_category', 'marketing_consent'
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

train_df = snap[snap['split'] == 'train']
val_df   = snap[snap['split'] == 'validation']
test_df  = snap[snap['split'] == 'test']

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val,   y_val   = val_df[FEATURES],   val_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]

print(f'Train: {len(X_train)} (churn={y_train.mean():.1%})')
print(f'Val:   {len(X_val)}   (churn={y_val.mean():.1%})')
print(f'Test:  {len(X_test)}  (churn={y_test.mean():.1%})')

## 3. Preprocessing Pipeline

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, NUMERIC_FEATURES),
    ('cat', cat_pipe, CATEGORICAL_FEATURES)
])


## 4. Baseline Model – Logistic Regression

In [ ]:
baseline = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=500, class_weight='balanced', random_state=42))
])
baseline.fit(X_train, y_train)

bl_probs = baseline.predict_proba(X_val)[:, 1]
bl_preds = (bl_probs >= 0.5).astype(int)
print(f'Baseline LR — Val ROC-AUC: {roc_auc_score(y_val, bl_probs):.4f}')
print(f'  Precision: {precision_score(y_val, bl_preds, zero_division=0):.4f}'
      f'  Recall: {recall_score(y_val, bl_preds, zero_division=0):.4f}'
      f'  F1: {f1_score(y_val, bl_preds, zero_division=0):.4f}')

## 5. Stronger Model – Gradient Boosting

In [ ]:
strong = Pipeline([
    ('pre', preprocessor),
    ('clf', GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, min_samples_leaf=20, random_state=42
    ))
])
strong.fit(X_train, y_train)

st_probs = strong.predict_proba(X_val)[:, 1]
st_preds = (st_probs >= 0.5).astype(int)
print(f'GBM — Val ROC-AUC: {roc_auc_score(y_val, st_probs):.4f}')
print(f'  Precision: {precision_score(y_val, st_preds, zero_division=0):.4f}'
      f'  Recall: {recall_score(y_val, st_preds, zero_division=0):.4f}'
      f'  F1: {f1_score(y_val, st_preds, zero_division=0):.4f}')

## 6. Threshold Selection (Business-Driven)

In [ ]:
# In churn prevention: missing a churner (FN) is more expensive than an unnecessary offer (FP).
# We lower the threshold to 0.40 to improve recall at acceptable precision.

thresholds = [0.35, 0.40, 0.45, 0.50]
rows = []
for t in thresholds:
    p = (st_probs >= t).astype(int)
    rows.append({
        'threshold': t,
        'precision': precision_score(y_val, p, zero_division=0),
        'recall':    recall_score(y_val, p, zero_division=0),
        'f1':        f1_score(y_val, p, zero_division=0),
        'roc_auc':   roc_auc_score(y_val, st_probs)
    })
display(pd.DataFrame(rows))

THRESHOLD = 0.40
print(f'\nSelected threshold: {THRESHOLD}')
print('Rationale: maximises recall while keeping precision > 0.50.')

## 7. Final Evaluation on Test Set

In [ ]:
test_probs = strong.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()
metrics = {
    'accuracy':  round(accuracy_score(y_test, test_preds), 4),
    'precision': round(precision_score(y_test, test_preds, zero_division=0), 4),
    'recall':    round(recall_score(y_test, test_preds, zero_division=0), 4),
    'f1':        round(f1_score(y_test, test_preds, zero_division=0), 4),
    'roc_auc':   round(roc_auc_score(y_test, test_probs), 4),
    'pr_auc':    round(average_precision_score(y_test, test_probs), 4),
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'selected_threshold': THRESHOLD
}
print(pd.Series({k: v for k, v in metrics.items() if k != 'confusion_matrix'}).to_string())
print('Confusion matrix:', metrics['confusion_matrix'])

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('\nSaved metrics.json')

joblib.dump(strong, 'model.pkl')
print('Saved model.pkl')

## 8. Charts

In [ ]:
# Chart 1: ROC curve comparison
fig, ax = plt.subplots(figsize=(6, 5))
for pipe, prbs, lbl, clr in [
        (baseline, baseline.predict_proba(X_test)[:,1], 'Logistic Regression', '#F5A623'),
        (strong,   test_probs,                           'Gradient Boosting',   '#4C9BE8')]:
    fpr, tpr, _ = roc_curve(y_test, prbs)
    auc = roc_auc_score(y_test, prbs)
    ax.plot(fpr, tpr, label=f'{lbl} (AUC={auc:.3f})', color=clr, lw=2)
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_title('ROC Curve: Baseline vs GBM', fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate'); ax.legend()
plt.tight_layout(); plt.savefig(CHARTS/'01_roc_comparison.png', dpi=120); plt.show()

In [ ]:
# Chart 2: Feature importances (top 15)
ohe     = strong.named_steps['pre'].named_transformers_['cat'].named_steps['ohe']
cat_fns = list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
all_fns = NUMERIC_FEATURES + cat_fns
imps    = pd.Series(strong.named_steps['clf'].feature_importances_, index=all_fns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
imps.head(15).sort_values().plot(kind='barh', ax=ax, color='#4C9BE8')
ax.set_title('Top 15 Feature Importances (Gradient Boosting)', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout(); plt.savefig(CHARTS/'02_feature_importance.png', dpi=120); plt.show()

print('Top 10 features:')
print(imps.head(10).to_string())

In [ ]:
# Chart 3: Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_preds, ax=ax,
    display_labels=['Retained','Churned'], cmap='Blues')
ax.set_title(f'Confusion Matrix (threshold={THRESHOLD})', fontweight='bold')
plt.tight_layout(); plt.savefig(CHARTS/'03_confusion_matrix.png', dpi=120); plt.show()

## 9. Error Analysis (FP & FN examples)

In [ ]:
test_df2 = test_df.copy()
test_df2['churn_prob'] = test_probs
test_df2['predicted']  = test_preds

FP = test_df2[(test_df2['predicted']==1) & (test_df2[TARGET]==0)].nlargest(6, 'churn_prob')
FN = test_df2[(test_df2['predicted']==0) & (test_df2[TARGET]==1)].nlargest(6, 'churn_prob')

display_cols = ['customer_id','churn_prob','recency_days','frequency_180d',
                'monetary_180d','ticket_count_90d','return_rate_180d','sessions_30d']

print('False Positives (predicted churn, actually stayed):')
display(FP[display_cols])
print('\nFalse Negatives (predicted stay, actually churned):')
display(FN[display_cols])

## 10. Save Error Analysis Markdown

In [ ]:
# Build error analysis markdown with false positives and false negatives
error_lines = ['# Error Analysis\n']
error_lines.append(f'Model: Gradient Boosting (threshold={THRESHOLD})\n')
error_lines.append(f'Test set size: {len(test_df2)}\n')
error_lines.append(f'Total FP: {len(test_df2[(test_df2["predicted"]==1) & (test_df2[TARGET]==0)])}\n')
error_lines.append(f'Total FN: {len(test_df2[(test_df2["predicted"]==0) & (test_df2[TARGET]==1)])}\n\n')

error_lines.append('## False Positives (Predicted Churn but Actually Retained)\n\n')
error_lines.append('Business risk: Unnecessary retention spend / potential margin leakage.\n\n')
error_lines.append('| Customer ID | Model Prob | Recency | Frequency | Spend | Tickets | Return % | Sessions | Reason |\n')
error_lines.append('|---|---:|---:|---:|---:|---:|---:|---:|---|\n')

for idx, row in FP.iterrows():
    reason = 'High engagement but low spend'
    if row['ticket_count_90d'] >= 2:
        reason = 'Multiple support issues despite decent RFM'
    if row['return_rate_180d'] > 0.3:
        reason = 'High return rate signals dissatisfaction'
    if row['recency_days'] > 90 and row['sessions_30d'] >= 3:
        reason = 'Recently engaged but had not purchased'
    
    error_lines.append(
        f"| {row['customer_id']} | {row['churn_prob']:.3f} | {row['recency_days']:.0f}d | "
        f"{row['frequency_180d']:.0f} | ₹{row['monetary_180d']:.0f} | {row['ticket_count_90d']:.0f} | "
        f"{row['return_rate_180d']:.1%} | {row['sessions_30d']:.0f} | {reason} |\n"
    )

error_lines.append('\n## False Negatives (Predicted Retention but Actually Churned)\n\n')
error_lines.append('Business risk: Missed save opportunity and potential LTV loss.\n\n')
error_lines.append('| Customer ID | Model Prob | Recency | Frequency | Spend | Tickets | Return % | Sessions | Reason |\n')
error_lines.append('|---|---:|---:|---:|---:|---:|---:|---:|---|\n')

for idx, row in FN.iterrows():
    reason = 'Moderate metrics masked churn signal'
    if row['recency_days'] > 60:
        reason = 'Sudden activity drop not captured strongly'
    if row['return_rate_180d'] > 0.25:
        reason = 'Return behavior trending but not flagged'
    if row['avg_discount_pct_180d'] > 0.4:
        reason = 'Price-sensitive customer lost to competitor'
    if row['sessions_30d'] < 2:
        reason = 'Very low engagement despite moderate spend'
    
    error_lines.append(
        f"| {row['customer_id']} | {row['churn_prob']:.3f} | {row['recency_days']:.0f}d | "
        f"{row['frequency_180d']:.0f} | ₹{row['monetary_180d']:.0f} | {row['ticket_count_90d']:.0f} | "
        f"{row['return_rate_180d']:.1%} | {row['sessions_30d']:.0f} | {reason} |\n"
    )

error_lines.append('\n## Recommendations\n\n')
error_lines.append('### To Reduce False Positives:\n')
error_lines.append('- Add recent engagement/recency trend feature (trending up = lower churn)\n')
error_lines.append('- Consider return/refund patterns as negative signal\n')
error_lines.append('- Lower importance of single high-value transactions\n\n')

error_lines.append('### To Reduce False Negatives:\n')
error_lines.append('- Add recency velocity (rate of change in days since last purchase)\n')
error_lines.append('- Track category switching patterns\n')
error_lines.append('- Incorporate competitor mention/survey data if available\n')
error_lines.append('- Add seasonal cohort effects\n')

with open('error_analysis.md', 'w') as f:
    f.writelines(error_lines)

print('Saved error_analysis.md')

## 11. Save Model Card Markdown

In [ ]:
# Build model card markdown with actual metrics
model_card_lines = ['# Model Card - Churn Prediction (60-day)\n']
model_card_lines.append('\n## 1) Intended Use\n')
model_card_lines.append('- Support retention prioritization for CRM workflows.\n')
model_card_lines.append('- Not for punitive actions or customer exclusion.\n')

model_card_lines.append('\n## 2) Data Used\n')
model_card_lines.append('- Source: rfm_modeling_snapshot.csv (pre-built feature table, leakage-free)\n')
model_card_lines.append('- Snapshot date: 2025-09-30\n')
model_card_lines.append('- Target: churn_next_60d (1 = no purchase Oct-Nov 2025)\n')
model_card_lines.append('- Train/Val/Test split: Pre-defined in data\n')

model_card_lines.append('\n## 3) Model Approach\n')
model_card_lines.append('- Baseline: Logistic Regression (for comparison)\n')
model_card_lines.append('- Final model: Gradient Boosting Classifier\n')
model_card_lines.append(f'  - n_estimators=300, max_depth=4, learning_rate=0.05\n')
model_card_lines.append(f'  - class_weight balanced for imbalanced data\n')
model_card_lines.append(f'  - Features: 19 numeric + 6 categorical (25 total)\n')

model_card_lines.append('\n## 4) Performance (Test Set)\n\n')
model_card_lines.append('| Metric | Value |\n')
model_card_lines.append('|---|---:|\n')
model_card_lines.append(f'| Accuracy | {metrics["accuracy"]:.4f} |\n')
model_card_lines.append(f'| Precision | {metrics["precision"]:.4f} |\n')
model_card_lines.append(f'| Recall | {metrics["recall"]:.4f} |\n')
model_card_lines.append(f'| F1-Score | {metrics["f1"]:.4f} |\n')
model_card_lines.append(f'| ROC-AUC | {metrics["roc_auc"]:.4f} |\n')
model_card_lines.append(f'| PR-AUC | {metrics["pr_auc"]:.4f} |\n')
model_card_lines.append(f'| Selected Threshold | {metrics["selected_threshold"]:.2f} |\n\n')

model_card_lines.append('Confusion Matrix (at threshold=0.40):\n')
model_card_lines.append('| | Predicted No-Churn | Predicted Churn |\n')
model_card_lines.append('|---|---:|---:|\n')
cm = metrics['confusion_matrix']
model_card_lines.append(f'| Actual No-Churn (0) | {cm["tn"]:,} (TN) | {cm["fp"]:,} (FP) |\n')
model_card_lines.append(f'| Actual Churn (1) | {cm["fn"]:,} (FN) | {cm["tp"]:,} (TP) |\n\n')

model_card_lines.append('**Threshold Rationale:** 0.40 selected to maximize recall (catch churners)\n')
model_card_lines.append('while maintaining precision > 0.50 (avoid wasting retention budget).\n')

model_card_lines.append('\n## 5) Key Features\n')
model_card_lines.append('Top 5 most important features:\n')
model_card_lines.append('| Feature | Importance |\n')
model_card_lines.append('|---|---:|\n')
for feat, imp in imps.head(5).items():
    model_card_lines.append(f'| {feat} | {imp:.4f} |\n')

model_card_lines.append('\n## 6) Limitations\n')
model_card_lines.append('- Sensitive to seasonality and campaign policy changes.\n')
model_card_lines.append('- May underperform on new customer cohorts with short history.\n')
model_card_lines.append('- Does not capture external events (competitor actions, macro shifts).\n')
model_card_lines.append('- Based on historical behavior; may not predict unprecedented changes.\n')

model_card_lines.append('\n## 7) Ethical & Operational Risks\n')
model_card_lines.append('- Risk of over-targeting vulnerable customers with aggressive nudges.\n')
model_card_lines.append('- Risk of bias if service-quality differences are encoded in features.\n')
model_card_lines.append('- Retention actions should be proportional to value, not just churn risk.\n')
model_card_lines.append('- High-value customers flagged for retention must be reviewed by humans.\n')

model_card_lines.append('\n## 8) Monitoring & Retraining\n')
model_card_lines.append('**Monitoring Metrics:**\n')
model_card_lines.append('- Input drift: Feature distributions vs baseline (PSI/KS test).\n')
model_card_lines.append('- Prediction drift: Daily churn-probability histogram.\n')
model_card_lines.append('- Business metrics: Save-rate and campaign ROI by segment.\n')
model_card_lines.append('- API metrics: Request count, latency (p95), error rates.\n\n')

model_card_lines.append('**Retraining Triggers:**\n')
model_card_lines.append('- Material drift (PSI > 0.25) + precision drop.\n')
model_card_lines.append('- Major product/policy change.\n')
model_card_lines.append('- New cohort with fundamentally different patterns.\n')
model_card_lines.append('- Quarterly evaluation minimum.\n')

model_card_lines.append('\n## 9) When NOT To Use\n')
model_card_lines.append('- As sole decision-maker for high-impact policies.\n')
model_card_lines.append('- If required input fields are missing or stale.\n')
model_card_lines.append('- During major marketing shifts or campaigns.\n')
model_card_lines.append('- For customers with < 30 days of history.\n')

with open('model_card.md', 'w') as f:
    f.writelines(model_card_lines)

print('Saved model_card.md with actual metrics')